In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.mixture import GaussianMixture

import warnings
warnings.filterwarnings('ignore')

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

In [2]:
campaign_table = pd.read_csv("../dunnhumby_The-Complete-Journey/campaign_table.csv")
campaign_desc = pd.read_csv("../dunnhumby_The-Complete-Journey/campaign_desc.csv")
coupon_redempt = pd.read_csv("../dunnhumby_The-Complete-Journey/coupon_redempt.csv")
coupon = pd.read_csv("../dunnhumby_The-Complete-Journey/coupon.csv")
transaction_data = pd.read_csv("../dunnhumby_The-Complete-Journey/transaction_data.csv")
product = pd.read_csv("../dunnhumby_The-Complete-Journey/product.csv")
causal = pd.read_csv("../dunnhumby_The-Complete-Journey/causal_data.csv")
segmented_data = pd.read_csv("../dunnhumby_The-Complete-Journey/segmented_df.csv")

In [3]:
product.head()

,PRODUCT_ID,MANUFACTURER,DEPARTMENT,BRAND,COMMODITY_DESC,SUB_COMMODITY_DESC,CURR_SIZE_OF_PRODUCT
0,25671,2,GROCERY,National,FRZN ICE,ICE - CRUSHED/CUBED,22 LB
1,26081,2,MISC. TRANS.,National,NO COMMODITY DESCRIPTION,NO SUBCOMMODITY DESCRIPTION,
2,26093,69,PASTRY,Private,BREAD,BREAD:ITALIAN/FRENCH,
3,26190,69,GROCERY,Private,FRUIT - SHELF STABLE,APPLE SAUCE,50 OZ
4,26355,69,GROCERY,Private,COOKIES/CONES,SPECIALTY COOKIES,14 OZ


In [4]:
product.drop(product[product['SUB_COMMODITY_DESC'].str.strip() == ''].index, inplace=True)

In [5]:
txn_product_merged = transaction_data.merge(
    product[
        [
            "PRODUCT_ID",
            "COMMODITY_DESC",
            "SUB_COMMODITY_DESC",
            "DEPARTMENT"
        ]
    ],
    on="PRODUCT_ID",
    how="left"
)

In [6]:
transaction_data.isnull().sum()

household_key        0
BASKET_ID            0
DAY                  0
PRODUCT_ID           0
QUANTITY             0
SALES_VALUE          0
STORE_ID             0
RETAIL_DISC          0
TRANS_TIME           0
WEEK_NO              0
COUPON_DISC          0
COUPON_MATCH_DISC    0
dtype: int64

In [7]:
txn_product_merged.shape

(2595732, 15)

In [8]:
txn_product_merged.isnull().sum()

household_key            0
BASKET_ID                0
DAY                      0
PRODUCT_ID               0
QUANTITY                 0
SALES_VALUE              0
STORE_ID                 0
RETAIL_DISC              0
TRANS_TIME               0
WEEK_NO                  0
COUPON_DISC              0
COUPON_MATCH_DISC        0
COMMODITY_DESC        7839
SUB_COMMODITY_DESC    7839
DEPARTMENT            7839
dtype: int64

In [9]:
txn_product_merged = txn_product_merged.dropna()

In [10]:
txn_product_merged.head()

,household_key,BASKET_ID,DAY,PRODUCT_ID,QUANTITY,SALES_VALUE,STORE_ID,RETAIL_DISC,TRANS_TIME,WEEK_NO,COUPON_DISC,COUPON_MATCH_DISC,COMMODITY_DESC,SUB_COMMODITY_DESC,DEPARTMENT
0,2375,26984851472,1,1004906,1,1.39,364,-0.60,1631,1,0.0,0.0,POTATOES,POTATOES RUSSET (BULK&BAG),PRODUCE
1,2375,26984851472,1,1033142,1,0.82,364,0.00,1631,1,0.0,0.0,ONIONS,ONIONS SWEET (BULK&BAG),PRODUCE
2,2375,26984851472,1,1036325,1,0.99,364,-0.30,1631,1,0.0,0.0,VEGETABLES - ALL OTHERS,CELERY,PRODUCE
3,2375,26984851472,1,1082185,1,1.21,364,0.00,1631,1,0.0,0.0,TROPICAL FRUIT,BANANAS,PRODUCE
4,2375,26984851472,1,8160430,1,1.50,364,-0.39,1631,1,0.0,0.0,ORGANICS FRUIT & VEGETABLES,ORGANIC CARROTS,PRODUCE


In [11]:
txn_product_merged["SUB_COMMODITY_DESC"].nunique()

2382

In [12]:
sub_commodity_purchases = (
    txn_product_merged
    .groupby(
        [
            "household_key",
            "SUB_COMMODITY_DESC"
        ]
    )
    .agg(
        total_quantity=("QUANTITY", "sum"),
        total_sales=("SALES_VALUE", "sum")
    )
    .reset_index()
)

In [13]:
sub_commodity_purchases.head()

,household_key,SUB_COMMODITY_DESC,total_quantity,total_sales
0,1,ADULT ANALGESICS,3,16.67
1,1,ADULT CEREAL,2,7.48
2,1,AEROSOL TOPPINGS,1,1.99
3,1,AIR CARE - AEROSOLS,1,2.99
4,1,AIR CARE - CANDLES,6,16.73


In [14]:
sub_commodity_purchases.shape

(614459, 4)

In [15]:
sub_commodity_purchases.isnull().sum()

household_key         0
SUB_COMMODITY_DESC    0
total_quantity        0
total_sales           0
dtype: int64

In [16]:
sub_commodity_purchases['SUB_COMMODITY_DESC'].value_counts()

SUB_COMMODITY_DESC
FLUID MILK WHITE ONLY             2392
BANANAS                           2105
SOFT DRINKS 12/18&15PK CAN CAR    2059
POTATO CHIPS                      2046
SHREDDED CHEESE                   2031
PREMIUM                           1984
SFT DRNK 2 LITER BTL CARB INCL    1976
MAINSTREAM WHITE BREAD            1960
EGGS - LARGE                      1937
CANDY BARS (SINGLES)(INCLUDING    1909
TORTILLA/NACHO CHIPS              1882
POURABLE SALAD DRESSINGS          1864
DAIRY CASE 100% PURE JUICE - O    1858
TOILET TISSUE                     1830
CONDENSED SOUP                    1802
SEMI-SOLID SALAD DRESSING MAY     1780
PRIMAL                            1775
SUGAR                             1775
IWS SINGLE CHEESE                 1773
SPICES & SEASONINGS               1773
HAMBURGER BUNS                    1743
MARGARINE: TUBS AND BOWLS         1740
POTATOES RUSSET (BULK&BAG)        1715
ALL FAMILY CEREAL                 1708
EGGS - X-LARGE                    1682
MAINST

In [17]:
sub_commodity_freq = (
    sub_commodity_purchases
    .groupby("SUB_COMMODITY_DESC")
    .size()
    .sort_values(ascending=True)
)

sub_commodity_freq.head(50)

SUB_COMMODITY_DESC
*ATH ACCES:TOWEL BARS/SOAP D      1
HANDHELD EQUIPMENT                1
VASES                             1
ARBOR/TRELLIS                     1
ARRANGEMENTS (NON-ROSE)           1
AROMA THERAPY OIL/GELS/CANDLES    1
ULTRA/SUPER GIN                   1
NUT SUPP-ESSENTIAL OILS           1
SOLAR GIFTS                       1
NUT SUPP-INDIVIDUAL               1
NUT SUPP-MISC                     1
DRYWALL REPAIR                    1
SMALL ACCESS LADIES GLOVES        1
SLUG BAITS                        1
CAKES INGREDIENTS                 1
CABINET ACCS: HINGES/KNOBS        1
HH GAMES                          1
LIGHTBULBS                        1
FTD SERVICE CHARGES               1
GARDEN WICKER                     1
CAMERAS                           1
SMOKE DETECTORS                   1
CAMERAS-35MM/APS                  1
LICENSE APPAREL GIRLS 7-16        1
EASTER LILY                       1
NUT SUPP-WELLNESS                 1
NOVELTY LIGHTS                    1
NUT SUPP-

In [18]:
# building the household-item matrix
valid_commodities = (
    sub_commodity_freq[
        sub_commodity_freq >= 20
    ].index
)

sub_commodity_purchases = (
    sub_commodity_purchases[
        sub_commodity_purchases["SUB_COMMODITY_DESC"]
        .isin(valid_commodities)
    ]
)

In [19]:
sub_commodity_purchases.shape

(610053, 4)

In [20]:
sub_commodity_purchases["SUB_COMMODITY_DESC"].nunique()

1677

In [21]:
#commodity dataset created

In [22]:
household_txn_agg = (
    txn_product_merged
    .groupby("household_key")
    .agg(
        total_sales=("SALES_VALUE", "sum"),
        total_quantity=("QUANTITY", "sum"),
        total_transactions=("BASKET_ID", "count"),
        unique_sub_commodities=("SUB_COMMODITY_DESC", "nunique")
    )
    .reset_index()
)

household_txn_agg.head()

,household_key,total_sales,total_quantity,total_transactions,unique_sub_commodities
0,1,4330.16,1997,1714,302
1,2,1954.34,834,714,299
2,3,2653.21,8540,921,219
3,4,1200.11,382,301,110
4,5,779.06,245,222,125


In [23]:
household_txn_agg.sort_values(by='unique_sub_commodities', ascending=False).loc[1073
                                                                                ]

household_key               1074.00
total_sales                 7769.22
total_quantity            142911.00
total_transactions          2345.00
unique_sub_commodities       405.00
Name: 1073, dtype: float64

In [24]:
household_txn_agg.sort_values(by='unique_sub_commodities', ascending=False).head(30)

,household_key,total_sales,total_quantity,total_transactions,unique_sub_commodities
1452,1453,21661.29,95743,6540,903
2321,2322,23646.92,992718,5675,882
2458,2459,20671.50,488789,6624,813
1900,1901,11462.24,60290,4048,770
2283,2284,17152.63,185015,4621,762
717,718,19299.86,869732,6782,723
327,328,17332.13,356334,4864,714
1994,1995,15332.43,427801,4465,699
399,400,18494.14,862525,4664,693
1851,1852,9800.02,900726,2634,683


In [25]:
sub_commodity_rank = (
    sub_commodity_purchases
    .sort_values(
        [
            "household_key",
            "total_quantity",
            "total_sales"
        ],
        ascending=[True, False, False]
    )
)

In [26]:
sub_commodity_rank.shape

(610053, 4)

In [27]:
sub_commodity_rank.head(20)

,household_key,SUB_COMMODITY_DESC,total_quantity,total_sales
188,1,ORANGES NAVELS ALL,63,37.44
102,1,FRUIT/BREAKFAST BREAD,56,157.05
26,1,BEANS GREEN: FS/WHL/CUT,56,53.98
222,1,PUDDINGS DRY,54,41.03
90,1,ENTREES,48,127.29
45,1,CANDY BAGS-CHOCOCLATE,48,83.57
76,1,DAIRY CASE 100% PURE JUICE - O,46,129.81
251,1,SOFT DRINKS 12/18&15PK CAN CAR,43,121.13
22,1,BANANAS,42,37.74
96,1,FLUID MILK WHITE ONLY,41,61.92


In [28]:
top_sub_commodities = (
    sub_commodity_rank
    .groupby("household_key")
    .head(5)
)
top_sub_commodities.head(15)

,household_key,SUB_COMMODITY_DESC,total_quantity,total_sales
188,1,ORANGES NAVELS ALL,63,37.44
102,1,FRUIT/BREAKFAST BREAD,56,157.05
26,1,BEANS GREEN: FS/WHL/CUT,56,53.98
222,1,PUDDINGS DRY,54,41.03
90,1,ENTREES,48,127.29
561,2,SOFT DRINKS 12/18&15PK CAN CAR,40,113.00
525,2,RAMEN NOODLES/RAMEN CUPS,32,3.79
409,2,FLUID MILK WHITE ONLY,22,42.61
590,2,TUNA,16,33.13
327,2,BEANS GREEN: FS/WHL/CUT,12,7.24


In [29]:
top_sub_commodities.shape

(12498, 4)

In [30]:
top_sub_commodity_list = (
    top_sub_commodities
    .groupby("household_key")
    ["SUB_COMMODITY_DESC"]
    .apply(list)
    .reset_index()
)

top_sub_commodity_list.head()

,household_key,SUB_COMMODITY_DESC
0,1,"[ORANGES NAVELS ALL, FRUIT/BREAKFAST BREAD, BE..."
1,2,"[SOFT DRINKS 12/18&15PK CAN CAR, RAMEN NOODLES..."
2,3,"[GASOLINE-REG UNLEADED, SFT DRNK 2 LITER BTL C..."
3,4,"[PIZZA/ECONOMY, CHEESE CRACKERS (CHEEZ-ITS/GOL..."
4,5,"[ISOTONIC DRINKS SINGLE SERVE, ISOTONIC DRINKS..."


In [31]:
top_sub_commodity_list['SUB_COMMODITY_DESC'][0]

['ORANGES NAVELS ALL',
 'FRUIT/BREAKFAST BREAD',
 'BEANS GREEN: FS/WHL/CUT',
 'PUDDINGS DRY',
 'ENTREES']

In [32]:
household_txn_agg = (
    household_txn_agg
    .merge(
        top_sub_commodity_list,
        on="household_key",
        how="left"
    )
)

household_txn_agg.head()

,household_key,total_sales,total_quantity,total_transactions,unique_sub_commodities,SUB_COMMODITY_DESC
0,1,4330.16,1997,1714,302,"[ORANGES NAVELS ALL, FRUIT/BREAKFAST BREAD, BE..."
1,2,1954.34,834,714,299,"[SOFT DRINKS 12/18&15PK CAN CAR, RAMEN NOODLES..."
2,3,2653.21,8540,921,219,"[GASOLINE-REG UNLEADED, SFT DRNK 2 LITER BTL C..."
3,4,1200.11,382,301,110,"[PIZZA/ECONOMY, CHEESE CRACKERS (CHEEZ-ITS/GOL..."
4,5,779.06,245,222,125,"[ISOTONIC DRINKS SINGLE SERVE, ISOTONIC DRINKS..."


In [33]:
household_txn_agg["sales_spend_tier"] = pd.qcut(
    household_txn_agg["total_sales"],
    q=4,
    labels=[
        "Low",
        "Medium",
        "High",
        "Premium"
    ]
)

In [34]:
household_txn_agg["quantity_tier"] = pd.qcut(
    household_txn_agg["total_quantity"],
    q=4,
    labels=[
        "Low",
        "Medium",
        "High",
        "Premium"
    ]
)

In [35]:
household_txn_agg.head()

,household_key,total_sales,total_quantity,total_transactions,unique_sub_commodities,SUB_COMMODITY_DESC,sales_spend_tier,quantity_tier
0,1,4330.16,1997,1714,302,"[ORANGES NAVELS ALL, FRUIT/BREAKFAST BREAD, BE...",High,Medium
1,2,1954.34,834,714,299,"[SOFT DRINKS 12/18&15PK CAN CAR, RAMEN NOODLES...",Medium,Medium
2,3,2653.21,8540,921,219,"[GASOLINE-REG UNLEADED, SFT DRNK 2 LITER BTL C...",High,Medium
3,4,1200.11,382,301,110,"[PIZZA/ECONOMY, CHEESE CRACKERS (CHEEZ-ITS/GOL...",Medium,Low
4,5,779.06,245,222,125,"[ISOTONIC DRINKS SINGLE SERVE, ISOTONIC DRINKS...",Low,Low


In [36]:
household_txn_agg.shape

(2500, 8)

In [37]:
household_txn_agg.iloc[0]["SUB_COMMODITY_DESC"]

['ORANGES NAVELS ALL',
 'FRUIT/BREAKFAST BREAD',
 'BEANS GREEN: FS/WHL/CUT',
 'PUDDINGS DRY',
 'ENTREES']

In [38]:
sub_commodity_purchases.head()

,household_key,SUB_COMMODITY_DESC,total_quantity,total_sales
0,1,ADULT ANALGESICS,3,16.67
1,1,ADULT CEREAL,2,7.48
2,1,AEROSOL TOPPINGS,1,1.99
3,1,AIR CARE - AEROSOLS,1,2.99
4,1,AIR CARE - CANDLES,6,16.73


In [39]:
sub_commodity_purchase_segment = (
    sub_commodity_purchases
    .merge(
        segmented_data,
        on="household_key",
        how="left"
    )
)

sub_commodity_purchase_segment.head()

,household_key,SUB_COMMODITY_DESC,total_quantity,total_sales,cluster
0,1,ADULT ANALGESICS,3,16.67,1
1,1,ADULT CEREAL,2,7.48,1
2,1,AEROSOL TOPPINGS,1,1.99,1
3,1,AIR CARE - AEROSOLS,1,2.99,1
4,1,AIR CARE - CANDLES,6,16.73,1


In [40]:
sub_commodity_purchase_segment.shape

(610053, 5)

In [41]:
sub_commodity_purchase_segment["cluster"].value_counts()

cluster
1    422624
0    175989
2     11440
Name: count, dtype: int64

In [42]:
segment_sub_commodity_sales_agg = (
    sub_commodity_purchase_segment
    .groupby(
        [
            "cluster",
            "SUB_COMMODITY_DESC"
        ]
    )
    .agg(
        total_sales=("total_sales", "sum"),
        total_quantity=("total_quantity", "sum"),
        households=("household_key", "nunique")
    )
    .reset_index()
)

In [43]:
segment_sub_commodity_sales_agg.head()

,cluster,SUB_COMMODITY_DESC,total_sales,total_quantity,households
0,0,ABRASIVES,497.68,303,146
1,0,ACNE MEDICATIONS,1137.55,208,87
2,0,ACTIVITY,304.60,82,50
3,0,ADDITIVES/FLUIDS,222.12,62,39
4,0,ADHESIVES/CAULK,215.33,97,60


In [44]:
segment_sub_commodity_sales_agg = (
    segment_sub_commodity_sales_agg
    .sort_values(
        [
            "cluster",
            "total_quantity",
            "total_sales"
        ],
        ascending=[True, False, False]
    )
)

In [45]:
segment_sub_commodity_sales_agg.head()

,cluster,SUB_COMMODITY_DESC,total_sales,total_quantity,households
648,0,GASOLINE-REG UNLEADED,470935.55,192835466,394
557,0,FLUID MILK WHITE ONLY,65961.87,37068,415
1676,0,YOGURT NOT MULTI-PACKS,14201.16,24142,357
1439,0,SOFT DRINKS 12/18&15PK CAN CAR,61788.07,21570,403
229,0,CANDY BARS (SINGLES)(INCLUDING,7795.00,20103,395


In [46]:
segment_top15 = (
    segment_sub_commodity_sales_agg
    .groupby("cluster")
    .head(15)
)

In [47]:
segment_top15.head(30)

,cluster,SUB_COMMODITY_DESC,total_sales,total_quantity,households
648,0,GASOLINE-REG UNLEADED,470935.55,192835466,394
557,0,FLUID MILK WHITE ONLY,65961.87,37068,415
1676,0,YOGURT NOT MULTI-PACKS,14201.16,24142,357
1439,0,SOFT DRINKS 12/18&15PK CAN CAR,61788.07,21570,403
229,0,CANDY BARS (SINGLES)(INCLUDING,7795.00,20103,395
1390,0,SFT DRNK 2 LITER BTL CARB INCL,19763.83,19643,387
1402,0,SHREDDED CHEESE,26586.34,13543,406
341,0,CONDENSED SOUP,13392.51,12967,395
113,0,BANANAS,12263.91,12506,409
861,0,MAINSTREAM WHITE BREAD,16128.70,12329,366


In [48]:
segment_popular_items = (
    segment_top15
    .groupby("cluster")
    ["SUB_COMMODITY_DESC"]
    .apply(list)
    .reset_index()
)

segment_popular_items.head()

,cluster,SUB_COMMODITY_DESC
0,0,"[GASOLINE-REG UNLEADED, FLUID MILK WHITE ONLY,..."
1,1,"[GASOLINE-REG UNLEADED, FLUID MILK WHITE ONLY,..."
2,2,"[GASOLINE-REG UNLEADED, FLUID MILK WHITE ONLY,..."


In [49]:
segment_scores = (
    segment_sub_commodity_sales_agg
    .copy()
)

segment_scores["segment_score"] = (
    segment_scores
    .groupby("cluster")["total_quantity"]
    .transform(
        lambda x:
        x / x.max()
    )
)

In [50]:
segment_popular_items

,cluster,SUB_COMMODITY_DESC
0,0,"[GASOLINE-REG UNLEADED, FLUID MILK WHITE ONLY,..."
1,1,"[GASOLINE-REG UNLEADED, FLUID MILK WHITE ONLY,..."
2,2,"[GASOLINE-REG UNLEADED, FLUID MILK WHITE ONLY,..."


In [51]:
segment_popular_items['SUB_COMMODITY_DESC'][0]

['GASOLINE-REG UNLEADED',
 'FLUID MILK WHITE ONLY',
 'YOGURT NOT MULTI-PACKS',
 'SOFT DRINKS 12/18&15PK CAN CAR',
 'CANDY BARS (SINGLES)(INCLUDING',
 'SFT DRNK 2 LITER BTL CARB INCL',
 'SHREDDED CHEESE',
 'CONDENSED SOUP',
 'BANANAS',
 'MAINSTREAM WHITE BREAD',
 'SS ECONOMY ENTREES/DINNERS ALL',
 'POTATO CHIPS',
 'PREMIUM',
 'SOFT DRINK POWDER POUCHES',
 'BEERALEMALT LIQUORS']

In [52]:
segment_scores.head(20)

,cluster,SUB_COMMODITY_DESC,total_sales,total_quantity,households,segment_score
648,0,GASOLINE-REG UNLEADED,470935.55,192835466,394,1.000000
557,0,FLUID MILK WHITE ONLY,65961.87,37068,415,0.000192
1676,0,YOGURT NOT MULTI-PACKS,14201.16,24142,357,0.000125
1439,0,SOFT DRINKS 12/18&15PK CAN CAR,61788.07,21570,403,0.000112
229,0,CANDY BARS (SINGLES)(INCLUDING,7795.00,20103,395,0.000104
1390,0,SFT DRNK 2 LITER BTL CARB INCL,19763.83,19643,387,0.000102
1402,0,SHREDDED CHEESE,26586.34,13543,406,0.000070
341,0,CONDENSED SOUP,13392.51,12967,395,0.000067
113,0,BANANAS,12263.91,12506,409,0.000065
861,0,MAINSTREAM WHITE BREAD,16128.70,12329,366,0.000064


In [53]:
#creating a collaborative matrix per house
#segment recommendation and household recommendation

1. Collabaritive matrix for our households
2. Find the similarity of the households
3. Caller function to get similar household for a particular households.
4. Get the products. 

In [54]:
sub_commodity_purchases.head()

,household_key,SUB_COMMODITY_DESC,total_quantity,total_sales
0,1,ADULT ANALGESICS,3,16.67
1,1,ADULT CEREAL,2,7.48
2,1,AEROSOL TOPPINGS,1,1.99
3,1,AIR CARE - AEROSOLS,1,2.99
4,1,AIR CARE - CANDLES,6,16.73


In [55]:
sub_commodity_purchases["SUB_COMMODITY_DESC"].nunique()

1677

In [56]:
sub_commodity_purchases['SUB_COMMODITY_DESC'].value_counts()

SUB_COMMODITY_DESC
FLUID MILK WHITE ONLY             2392
BANANAS                           2105
SOFT DRINKS 12/18&15PK CAN CAR    2059
POTATO CHIPS                      2046
SHREDDED CHEESE                   2031
PREMIUM                           1984
SFT DRNK 2 LITER BTL CARB INCL    1976
MAINSTREAM WHITE BREAD            1960
EGGS - LARGE                      1937
CANDY BARS (SINGLES)(INCLUDING    1909
TORTILLA/NACHO CHIPS              1882
POURABLE SALAD DRESSINGS          1864
DAIRY CASE 100% PURE JUICE - O    1858
TOILET TISSUE                     1830
CONDENSED SOUP                    1802
SEMI-SOLID SALAD DRESSING MAY     1780
SUGAR                             1775
PRIMAL                            1775
IWS SINGLE CHEESE                 1773
SPICES & SEASONINGS               1773
HAMBURGER BUNS                    1743
MARGARINE: TUBS AND BOWLS         1740
POTATOES RUSSET (BULK&BAG)        1715
ALL FAMILY CEREAL                 1708
EGGS - X-LARGE                    1682
KIDS C

In [57]:
household_item_matrix = sub_commodity_purchases.pivot_table(index = 'household_key', columns = 'SUB_COMMODITY_DESC', values = 'total_quantity').fillna(0)

In [58]:
household_item_matrix.head()

SUB_COMMODITY_DESC  ABRASIVES  ACNE MEDICATIONS  ACTIVITY  ADDITIVES/FLUIDS  \
household_key                                                                 
1                         0.0               0.0       0.0               0.0   
2                         0.0               0.0       0.0               0.0   
3                         0.0               0.0       0.0               0.0   
4                         0.0               0.0       0.0               0.0   
5                         0.0               1.0       0.0               0.0   

SUB_COMMODITY_DESC  ADHESIVES/CAULK  ADULT ANALGESICS  ADULT CEREAL  \
household_key                                                         
1                               0.0               3.0           2.0   
2                               0.0               0.0           0.0   
3                               0.0               0.0           4.0   
4                               0.0               1.0           0.0   
5                               0.0               0.0           0.0   

SUB_COMMODITY_DESC  ADULT INCONTINENCE BRIEFS  ADULT INCONTINENCE MISC PRODUC  \
household_key                                                                   
1                                         0.0                             0.0   
2                                         0.0                             1.0   
3                                         0.0                             0.0   
4                                         0.0                             0.0   
5                                         0.0                             0.0   

SUB_COMMODITY_DESC  ADULT INCONTINENCE PADS  ADULT INCONTINENCE UNDERGARMEN  \
household_key                                                                 
1                                       0.0                             0.0   
2                                       0.0                             0.0   
3                                       0.0                             0.0   
4                                       0.0                             0.0   
5                                       0.0                             0.0   

SUB_COMMODITY_DESC  ADULT PREMIUM  AEROSOL DEODORANTS  AEROSOL TOPPINGS  \
household_key                                                             
1                             0.0                 0.0               1.0   
2                             0.0                 0.0               0.0   
3                             0.0                 0.0               0.0   
4                             0.0                 0.0               0.0   
5                             0.0                 1.0               0.0   

SUB_COMMODITY_DESC  AGE RESTRICTED DVD S  AGE RESTRICTED FIREWORKS  \
household_key                                                        
1                                    0.0                       0.0   
2                                    0.0                       0.0   
3                                    0.0                       0.0   
4                                    0.0                       0.0   
5                                    0.0                       0.0   

SUB_COMMODITY_DESC  AGE RESTRICTED SPHE $9.99  AIR CARE - AEROSOLS  \
household_key                                                        
1                                         0.0                  1.0   
2                                         0.0                  0.0   
3                                         0.0                  2.0   
4                                         0.0                  4.0   
5                                         0.0                  0.0   

SUB_COMMODITY_DESC  AIR CARE - CANDLES  AIR CARE - CONTINUOUS - NON EL  \
household_key                                                            
1                                  6.0                             6.0   
2                                  0.0                             0.0   
3                     

In [59]:
household_item_matrix.shape

(2500, 1677)

In [60]:
from sklearn.metrics.pairwise import cosine_similarity
similarity_matrix = cosine_similarity(household_item_matrix)
similarity_matrix

array([[1.00000000e+00, 3.76361471e-01, 4.28902872e-03, ...,
        1.16478745e-03, 3.21663373e-04, 2.10165628e-04],
       [3.76361471e-01, 1.00000000e+00, 9.12665269e-03, ...,
        1.28083167e-03, 4.66291642e-04, 2.97045944e-04],
       [4.28902872e-03, 9.12665269e-03, 1.00000000e+00, ...,
        9.99572543e-01, 9.99562603e-01, 9.99553959e-01],
       ...,
       [1.16478745e-03, 1.28083167e-03, 9.99572543e-01, ...,
        1.00000000e+00, 9.99995499e-01, 9.99995372e-01],
       [3.21663373e-04, 4.66291642e-04, 9.99562603e-01, ...,
        9.99995499e-01, 1.00000000e+00, 9.99999290e-01],
       [2.10165628e-04, 2.97045944e-04, 9.99553959e-01, ...,
        9.99995372e-01, 9.99999290e-01, 1.00000000e+00]],
      shape=(2500, 2500))

In [61]:
similarity_df = pd.DataFrame(similarity_matrix, index = household_item_matrix.index, columns = household_item_matrix.index)
similarity_df.head()

household_key      1         2         3         4         5         6     \
household_key                                                               
1              1.000000  0.376361  0.004289  0.175924  0.112424  0.001180   
2              0.376361  1.000000  0.009127  0.200293  0.244808  0.000875   
3              0.004289  0.009127  1.000000  0.005252  0.003987  0.999560   
4              0.175924  0.200293  0.005252  1.000000  0.114943  0.000224   
5              0.112424  0.244808  0.003987  0.114943  1.000000  0.000330   

household_key      7         8         9         10        11        12    \
household_key                                                               
1              0.367807  0.001010  0.223489  0.103201  0.278010  0.170172   
2              0.255791  0.001636  0.292787  0.180351  0.230052  0.154872   
3              0.009104  0.999573  0.008540  0.011622  0.002501  0.003870   
4              0.137137  0.000835  0.190672  0.141342  0.030105  0.071800   
5              0.141334  0.000686  0.107686  0.052773  0.079375  0.086731   

household_key      13        14        15        16        17        18    \
household_key                                                               
1              0.000077  0.000256  0.239660  0.000250  0.000196  0.000301   
2              0.000093  0.000510  0.302521  0.000286  0.000439  0.000227   
3              0.999550  0.999563  0.005440  0.999555  0.999551  0.999553   
4              0.000052  0.000241  0.138748  0.000169  0.000136  0.000171   
5              0.000053  0.000183  0.176736  0.000151  0.000032  0.000384   

household_key      19        20        21        22        23        24    \
household_key                                                               
1              0.000364  0.000263  0.000450  0.000232  0.000144  0.223181   
2              0.000511  0.000285  0.000922  0.000155  0.000167  0.407857   
3              0.999561  0.999553  0.999561  0.999552  0.999554  0.015237   
4              0.000256  0.000189  0.000562  0.000098  0.000173  0.324410   
5              0.000226  0.000131  0.000338  0.000068  0.000151  0.119731   

household_key      25        26        27        28        29        30    \
household_key                                                               
1              0.001628  0.195326  0.046628  0.000743  0.099014  0.000109   
2              0.001973  0.197341  0.083124  0.000942  0.142700  0.000159   
3              0.999584  0.008839  0.005112  0.999564  0.007935  0.999551   
4              0.001194  0.149929  0.161285  0.000679  0.503653  0.000095   
5              0.001160  0.110201  0.075744  0.000241  0.151087  0.000075   

household_key      31        32        33        34        35        36    \
household_key                                                               
1              0.000136  0.000105  0.000389  0.000293  0.000355  0.007343   
2              0.000156  0.000112  0.000767  0.000551  0.000552  0.013439   
3              0.999552  0.999551  0.999564  0.999585  0.999561  0.998999   
4              0.000081  0.000059  0.000407  0.000345  0.000208  0.010861   
5              0.000062  0.000055  0.000369  0.000205  0.000224  0.005500   

household_key      37        38        39        40        41        42    \
household_key                                                               
1              0.000173  0.233591  0.004187  0.000080  0.153986  0.131665   
2              0.000869  0.173703  0.005119  0.000113  0.217338  0.134286   
3              0.999569  0.004132  0.999611  0.999550  0.002300  0.003334   
4              0.000054  0.053131  0.004121  0.000038  0.058171  0.117849   
5              0.000081  0.029174  0.003118  0.000050  0.077369  0.072912   

household_key      43        44        45        46        47        48    \
household_key                                                               
1              0.000149  0.255290  0.102224  0.000910  0.301626  0

In [62]:
# cosine similarity
def get_similar_households(household_id):
    similarities = similarity_df.loc[household_id].sort_values(ascending = False).iloc[1:6]
    return similarities

In [63]:
get_similar_households(1)

household_key
1064    0.463453
1590    0.457699
1719    0.445255
1908    0.432980
2181    0.431268
Name: 1, dtype: float64

In [64]:
get_similar_households(2000)

household_key
407    0.669443
358    0.613822
136    0.599582
930    0.598569
290    0.598268
Name: 2000, dtype: float64

In [65]:
get_similar_households(17)

household_key
176     0.999999
588     0.999999
963     0.999999
2093    0.999999
1299    0.999999
Name: 17, dtype: float64

In [66]:
get_similar_households(18)

household_key
1612    1.0
1944    1.0
949     1.0
744     1.0
2311    1.0
Name: 18, dtype: float64

In [67]:
# top 5 recs per household
count = 0
for i in range(1,2501):
    
    val = get_similar_households(i).iloc[0]
    if val > 0.9:
        count += 1
        print(val, count)
        


0.9997023580992275 1
0.9999964655400603 2
0.9999967453403237 3
0.9999999802793325 4
0.999999637852833 5
0.9999996760676475 6
0.9999994694367884 7
0.9999995834378735 8
0.9999997817891497 9
0.9999996891497898 10
0.9999979303783336 11
0.999999911681041 12
0.9999997833054977 13
0.9999927105248819 14
0.9999969151640152 15
0.9999999228597438 16
0.9999999544523995 17
0.9999999539746749 18
0.9999978592606531 19
0.9999966411410244 20
0.9999995019556078 21
0.9995299802639839 22
0.9999857919830529 23
0.9998940580874885 24
0.999999942378852 25
0.9999985627186357 26
0.9999982348614996 27
0.9999989175846674 28
0.9999997340396121 29
0.9999997505289693 30
0.9999992838801889 31
0.9995310440834096 32
0.999998581139147 33
0.9999999806282766 34
0.9999986349147995 35
0.9999991051425899 36
0.999999697297452 37
0.9999991235090102 38
0.9999640499690038 39
0.9999990827393277 40
0.9999999258983594 41
0.9999999753105941 42
0.9999998295388044 43
0.9999995389589914 44
0.999999035418162 45
0.9999673060845784 46
0.9

In [68]:
get_similar_households(17)

household_key
176     0.999999
588     0.999999
963     0.999999
2093    0.999999
1299    0.999999
Name: 17, dtype: float64

In [69]:
sub_commodity_purchases.head(10)

,household_key,SUB_COMMODITY_DESC,total_quantity,total_sales
0,1,ADULT ANALGESICS,3,16.67
1,1,ADULT CEREAL,2,7.48
2,1,AEROSOL TOPPINGS,1,1.99
3,1,AIR CARE - AEROSOLS,1,2.99
4,1,AIR CARE - CANDLES,6,16.73
5,1,AIR CARE - CONTINUOUS - NON EL,6,14.47
6,1,AIR CARE - CONTINUOUS ACTION,9,23.47
7,1,ALL FAMILY CEREAL,14,41.11
8,1,ALUMINUM FOIL,4,10.33
9,1,ANTIPERSPIRANTS ONLY (ALL OTHE,5,18.75


In [70]:
def top5_recommendations(household_id):
    similar_households = get_similar_households(household_id)
    similar_households = similar_households.index
    similar_household_purchases = top_sub_commodities[top_sub_commodities["household_key"]
                                                      .isin(similar_households)][['household_key','SUB_COMMODITY_DESC', 
                                                                                  'total_quantity']]
    similar_household_purchases = similar_household_purchases.groupby(["SUB_COMMODITY_DESC"])["total_quantity"].sum().sort_values(ascending = False).reset_index().head(5)
    five_best_recommendations = similar_household_purchases["SUB_COMMODITY_DESC"].values
    for i in five_best_recommendations:
        print(i)

In [71]:
top5_recommendations(200)

SFT DRNK SNGL SRV BTL CARB (EX
CANDY BARS (SINGLES)(INCLUDING
SGL SV/VEND MACH SNACKS CHIP/P
SFT DRNK 2 LITER BTL CARB INCL
SFT DRNK MLT-PK BTL CARB (EXCP
